# Example 2: 4-bit Quantization

This notebook demonstrates advanced 4-bit quantization techniques:
- Basic 4-bit quantization
- Block-wise quantization
- NF4 (Normal Float 4) quantization
- Comparison with 8-bit quantization

**Key Benefit:** 4-bit achieves **8× memory reduction** vs FP32!

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
from typing import Tuple

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Simple 4-bit Quantization

4-bit allows values from -8 to 7 (signed) or 0 to 15 (unsigned).

In [ ]:
def quantize_4bit_simple(tensor: torch.Tensor) -> Tuple[torch.Tensor, float, float]:
    """Simple 4-bit quantization (signed)."""
    qmin, qmax = -8, 7
    
    min_val = tensor.min()
    max_val = tensor.max()
    
    scale = (max_val - min_val) / (qmax - qmin)
    zero_point = qmin - torch.round(min_val / scale)
    
    quantized = torch.clamp(torch.round(tensor / scale) + zero_point, qmin, qmax)
    
    return quantized.to(torch.int8), scale.item(), zero_point.item()

def dequantize_4bit_simple(quantized: torch.Tensor, scale: float, zero_point: float) -> torch.Tensor:
    return (quantized.float() - zero_point) * scale

# Test
tensor = torch.randn(1000, 1000, device=device)
q_4bit, scale, zp = quantize_4bit_simple(tensor)
dq_4bit = dequantize_4bit_simple(q_4bit, scale, zp)

print(f"Original memory: {tensor.element_size() * tensor.nelement() / 1e6:.2f} MB")
print(f"4-bit memory (packed): {q_4bit.element_size() * q_4bit.nelement() / 2 / 1e6:.2f} MB")
print(f"Memory reduction: 8x")

mse = torch.mean((tensor - dq_4bit) ** 2).item()
print(f"Reconstruction MSE: {mse:.6f}")

## Part 2: Block-wise Quantization

Divide tensor into blocks and quantize each independently for better accuracy.

In [ ]:
def quantize_4bit_blockwise(tensor: torch.Tensor, block_size: int = 64):
    """Block-wise 4-bit quantization."""
    original_shape = tensor.shape
    tensor_flat = tensor.flatten()
    
    # Pad to block size
    pad_len = (block_size - len(tensor_flat) % block_size) % block_size
    if pad_len > 0:
        tensor_flat = torch.cat([tensor_flat, torch.zeros(pad_len, device=tensor.device)])
    
    # Reshape into blocks
    num_blocks = len(tensor_flat) // block_size
    tensor_blocks = tensor_flat.reshape(num_blocks, block_size)
    
    qmin, qmax = -8, 7
    quantized_blocks = []
    scales = []
    zero_points = []
    
    for block in tensor_blocks:
        min_val = block.min()
        max_val = block.max()
        
        scale = (max_val - min_val) / (qmax - qmin)
        if scale == 0:
            scale = 1.0
        
        zero_point = qmin - torch.round(min_val / scale)
        q_block = torch.clamp(torch.round(block / scale) + zero_point, qmin, qmax)
        
        quantized_blocks.append(q_block)
        scales.append(scale)
        zero_points.append(zero_point)
    
    quantized = torch.stack(quantized_blocks).flatten()
    if pad_len > 0:
        quantized = quantized[:-pad_len]
    
    return quantized.reshape(original_shape).to(torch.int8), torch.tensor(scales), torch.tensor(zero_points)

# Compare block sizes
tensor = torch.randn(1000, 1000, device=device)

for block_size in [16, 64, 256]:
    q_block, scales, zps = quantize_4bit_blockwise(tensor, block_size)
    print(f"\nBlock size {block_size}:")
    print(f"  Number of blocks: {len(scales)}")
    print(f"  Metadata overhead: {(scales.element_size() + zps.element_size()) * len(scales) / 1e3:.2f} KB")

## Part 3: NF4 (Normal Float 4) Quantization

NF4 is optimized for normally distributed data (neural network weights).  
It uses non-uniform quantization with more points near zero.

In [ ]:
# NF4 lookup table (optimized for normal distribution)
NF4_QUANT_TABLE = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
], dtype=torch.float32)

def quantize_nf4(tensor: torch.Tensor) -> Tuple[torch.Tensor, float]:
    """NF4 quantization."""
    absmax = tensor.abs().max()
    if absmax == 0:
        absmax = 1.0
    
    normalized = tensor / absmax
    quant_table = NF4_QUANT_TABLE.to(tensor.device)
    
    # Find nearest value
    normalized_expanded = normalized.unsqueeze(-1)
    quant_table_expanded = quant_table.view(*([1] * len(normalized.shape)), -1)
    
    distances = torch.abs(normalized_expanded - quant_table_expanded)
    indices = torch.argmin(distances, dim=-1)
    
    return indices.to(torch.uint8), absmax.item()

def dequantize_nf4(indices: torch.Tensor, absmax: float) -> torch.Tensor:
    quant_table = NF4_QUANT_TABLE.to(indices.device)
    normalized = quant_table[indices.long()]
    return normalized * absmax

# Test on normally distributed data
normal_tensor = torch.randn(1000, 1000, device=device) * 0.02

# Simple 4-bit
q_simple, scale_s, zp_s = quantize_4bit_simple(normal_tensor)
dq_simple = dequantize_4bit_simple(q_simple, scale_s, zp_s)
mse_simple = torch.mean((normal_tensor - dq_simple) ** 2).item()

# NF4
q_nf4, absmax = quantize_nf4(normal_tensor)
dq_nf4 = dequantize_nf4(q_nf4, absmax)
mse_nf4 = torch.mean((normal_tensor - dq_nf4) ** 2).item()

print(f"Simple 4-bit MSE: {mse_simple:.6f}")
print(f"NF4 MSE: {mse_nf4:.6f}")
print(f"\n✓ NF4 reduced error by {(1 - mse_nf4/mse_simple)*100:.1f}%!")

## Summary

### Key Takeaways:
1. ✅ 4-bit achieves **8× memory reduction**
2. ✅ Block-wise quantization preserves more information
3. ✅ NF4 is optimal for normally distributed weights
4. ✅ Smaller blocks = better accuracy but more overhead
5. ✅ NF4 is used in **QLoRA** for efficient fine-tuning

### Next: Run notebook 03 for real LLM quantization!